# Training many seeds and configurations in parallel with `jax.vmap`

<a href="https://colab.research.google.com/github/d-tiapkin/gfnx/blob/main/notebooks/02_vmap_seeds_and_configs.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In `gfnx` the environment and the reward are stateless and every array lives in a pytree,
so a whole training run — initialisation, rollouts, updates and evaluation — is a **pure
function of its inputs**. Pure functions can be vmapped. This notebook trains a
Trajectory Balance sampler on a hypergrid for a grid of random seeds × learning rates, in
a single compiled program. It takes a few minutes on a CPU runtime; a GPU runtime
(Runtime → Change runtime type) makes the parallel grid much cheaper.

The same pattern is used by `baselines/tb_hypergrid_multiseed.py` and
`baselines/db_hypergrid_multiseed.py`; the
[walkthrough](https://gfnx.readthedocs.io/en/latest/walkthrough/#vmapping-training-over-seeds-proof-of-concept)
explains it step by step.

## Setup

`gfnx` supports Python 3.12–3.14. On Colab, the cell below installs it from GitHub. If
the install upgrades `numpy` or `jax` while an older version is already loaded, it
restarts the runtime once: then run the notebook again from the top (the install is
skipped the second time).

In [ ]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("gfnx") is None:
    if not (3, 12) <= sys.version_info[:2] <= (3, 14):
        raise RuntimeError(f"gfnx supports Python 3.12-3.14, this runtime has {sys.version}.")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "gfnx @ git+https://github.com/d-tiapkin/gfnx.git",
            "equinox",
            "optax",
        ],
        check=True,
    )
    from importlib.metadata import version

    stale = [
        name
        for name in ("numpy", "jax")
        if name in sys.modules and sys.modules[name].__version__ != version(name)
    ]
    if stale and "google.colab" in sys.modules:
        print(f"Upgraded {stale}. Restarting the runtime: run the notebook again from the top.")
        os.kill(os.getpid(), 9)

In [ ]:
import time

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

import gfnx

print("jax", jax.__version__, "| devices:", jax.devices())

## Environment, reward, policy and loss

The same ingredients as in the quickstart notebook: a 2-D hypergrid with the "easy"
reward, an MLP forward policy with a uniform backward policy, and the TB loss.

In [ ]:
DIM, SIDE = 2, 16
env = gfnx.HypergridEnvironment(dim=DIM, side=SIDE)
env_params = env.init(jax.random.PRNGKey(0))
reward_module = gfnx.EasyHypergridRewardModule(side=SIDE)
reward_params = reward_module.init(jax.random.PRNGKey(0), env.reset())


class Policy(eqx.Module):
    mlp: eqx.nn.MLP

    def __init__(self, key):
        self.mlp = eqx.nn.MLP(
            in_size=env.observation_space.shape[0],
            out_size=env.action_space.n,
            width_size=128,
            depth=2,
            key=key,
        )

    def __call__(self, obs):
        return {
            "forward_logits": self.mlp(obs),
            "backward_logits": jnp.zeros(env.backward_action_space.n),  # uniform P_B
        }


# The static part (activation functions, shapes) is identical for every seed; only the
# arrays differ, so we split it off once.
_, model_static = eqx.partition(Policy(jax.random.PRNGKey(0)), eqx.is_array)


def policy_fn(rng_key, obs, model_params):
    out = eqx.combine(model_params, model_static)(obs)
    return out["forward_logits"], out


def tb_loss(params, traj_data, log_rewards):
    net = eqx.combine(params["model"], model_static)
    traj_data = traj_data.replace(info=jax.vmap(jax.vmap(net))(traj_data.obs))
    log_pf, log_pb = jax.vmap(
        lambda traj: gfnx.utils.forward_trajectory_log_probs(env, traj, env_params)
    )(traj_data)
    return jnp.mean((params["logZ"] + log_pf - log_rewards - log_pb) ** 2)


metric = gfnx.metrics.ExactDistributionMetricsModule(
    metrics=["tv"],
    env=env,
    fwd_policy_fn=policy_fn,
    reward_module=reward_module,
    batch_size=256,
)
metric_state = metric.init(
    jax.random.PRNGKey(0), metric.InitArgs(env_params=env_params, reward_params=reward_params)
)

## A training run as a pure function

`run(seed, learning_rate)` builds the model and the optimizer from its arguments, trains
for `NUM_STEPS` steps and returns the TV distance to the target after every
`EVAL_EVERY` steps. There is no Python-side state and no logging callback inside, which is
what makes it vmappable (`jax.debug.callback` is not). The learning rate is an ordinary
traced array, so it can be vmapped too.

In [ ]:
BATCH_SIZE = 64
NUM_STEPS = 2_000
EVAL_EVERY = 250
NUM_EVALS = NUM_STEPS // EVAL_EVERY
epsilon_schedule = optax.linear_schedule(1.0, 0.0, transition_steps=NUM_STEPS // 2)


def run(seed, learning_rate):
    init_key, train_key = jax.random.split(jax.random.PRNGKey(seed))
    model_params, _ = eqx.partition(Policy(init_key), eqx.is_array)
    params = {"model": model_params, "logZ": jnp.array(0.0)}
    optimizer = optax.multi_transform(
        {"net": optax.adam(learning_rate), "logZ": optax.adam(1e-1)},
        {"model": "net", "logZ": "logZ"},
    )
    opt_state = optimizer.init(params)

    def train_step(carry, inputs):
        params, opt_state = carry
        key, step = inputs
        epsilon = epsilon_schedule(step)

        def exploration_policy(rng_key, obs, model_params):
            logits, out = policy_fn(rng_key, obs, model_params)
            return gfnx.utils.apply_epsilon_greedy(rng_key, logits, epsilon), out

        traj_data, final_states, _ = jax.vmap(
            lambda k: gfnx.utils.forward_rollout(
                k, exploration_policy, params["model"], env, env_params
            )
        )(jax.random.split(key, BATCH_SIZE))
        log_rewards = jax.vmap(reward_module.log_reward, in_axes=(0, None))(
            final_states, reward_params
        )
        grads = jax.grad(tb_loss)(params, traj_data, log_rewards)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        return (optax.apply_updates(params, updates), opt_state), None

    def train_then_evaluate(carry, epoch):
        params, opt_state = carry
        steps = epoch * EVAL_EVERY + jnp.arange(EVAL_EVERY)
        keys = jax.random.split(jax.random.fold_in(train_key, epoch), EVAL_EVERY)
        (params, opt_state), _ = jax.lax.scan(train_step, (params, opt_state), (keys, steps))
        processed = metric.process(
            metric_state,
            jax.random.PRNGKey(0),
            metric.ProcessArgs(policy_params=params["model"], env_params=env_params),
        )
        return (params, opt_state), metric.get(processed)["tv"]

    _, tv_history = jax.lax.scan(train_then_evaluate, (params, opt_state), jnp.arange(NUM_EVALS))
    return tv_history  # [NUM_EVALS]

## One run

`jax.jit(run)` compiles the full training loop, then executes it without returning to
Python. Compiling ahead of time with `.lower(...).compile()` keeps compilation out of the
timing.

In [ ]:
run_single = jax.jit(run).lower(0, 1e-3).compile()

start = time.perf_counter()
tv_single = jax.block_until_ready(run_single(0, 1e-3))
single_time = time.perf_counter() - start
print(f"one run: {single_time:.1f} s, final TV {float(tv_single[-1]):.3f}")

## A grid of seeds × learning rates

Two nested `jax.vmap`s turn `run(seed, lr)` into a function of a whole grid. Each run
still sees its own seed and learning rate; XLA simply executes them side by side.

In [ ]:
seeds = jnp.arange(3)
learning_rates = jnp.array([3e-4, 1e-3, 3e-3])

run_grid = (
    jax.jit(
        jax.vmap(  # over learning rates
            jax.vmap(run, in_axes=(0, None)),  # over seeds
            in_axes=(None, 0),
        )
    )
    .lower(seeds, learning_rates)
    .compile()
)

start = time.perf_counter()
tv_grid = jax.block_until_ready(run_grid(seeds, learning_rates))  # [n_lr, n_seeds, NUM_EVALS]
grid_time = time.perf_counter() - start

n_runs = seeds.size * learning_rates.size
print(
    f"{n_runs} runs in {grid_time:.1f} s "
    f"({grid_time / n_runs:.2f} s per run vs {single_time:.1f} s for a single run)"
)

Mean ± standard deviation over seeds, for each learning rate:

In [ ]:
steps = EVAL_EVERY * (jnp.arange(NUM_EVALS) + 1)
for lr, tv in zip(learning_rates, tv_grid):
    mean, std = tv.mean(axis=0), tv.std(axis=0)
    plt.plot(steps, mean, label=f"lr = {float(lr):g}")
    plt.fill_between(steps, mean - std, mean + std, alpha=0.2)
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("TV distance to R(x)/Z")
plt.legend()
plt.show()

for lr, tv in zip(learning_rates, tv_grid):
    final = tv[:, -1]
    print(f"lr = {float(lr):g}: final TV {float(final.mean()):.3f} ± {float(final.std()):.3f}")

## What can and cannot go inside the vmap

- **Can:** anything that is a JAX array — model parameters, optimizer states, reward
  parameters, random keys, and scalar hyperparameters such as learning rates or the
  exploration rate.
- **Cannot:** Python-side effects (`jax.debug.callback`, printing, `wandb` logging) and
  anything that changes array *shapes* (network width, batch size, grid size). Sweep those
  with an ordinary Python loop around the vmapped function.

On a GPU, the vmapped grid typically costs little more than a single run until the device
is saturated.